In [1]:
import pandas as pd
import numpy as np
import os
import re
import gc
import handy.my_utils as deniz
from EEGPreprocessor import EEGPreprocessor
import mne

In [ ]:
CHB_MIT_PATH = 'data-understanding/data/chb-mit'
preprocessor = EEGPreprocessor(sampling_rate=256, target_sfreq=128)
new_fs = preprocessor.target_sfreq
all_annotations = {}
patient_dirs = sorted([d for d in os.listdir(CHB_MIT_PATH)
                       if os.path.isdir(os.path.join(CHB_MIT_PATH, d))
                       and d.startswith('chb')])

total_seizures = 0
total_files_with_seizures = 0

print(f"\n👥 Processing {len(patient_dirs)} patients...\n")

for patient in patient_dirs:
    patient_path = os.path.join(CHB_MIT_PATH, patient)
    summary_file = os.path.join(patient_path, f'{patient}-summary.txt')

    if os.path.exists(summary_file):
        seizure_info = deniz.parse_summary_file(summary_file)

        if seizure_info:
            all_annotations[patient] = seizure_info
            n_files = len(seizure_info)
            n_seizures = sum(len(times) for times in seizure_info.values())
            total_files_with_seizures += n_files
            total_seizures += n_seizures

            print(f"   ✓ {patient}: {n_files} files, {n_seizures} seizures")
        else:
            print(f"   - {patient}: No seizures found")
    else:
        print(f"   ✗ {patient}: No summary file")

In [ ]:
all_annotations

In [ ]:
import pandas as pd

# Boş listeler oluşturalım
patients = []
files = []
seizure_starts = []
seizure_ends = []

# İç içe geçmiş yapıyı düzleştirelim
for patient, patient_data in all_annotations.items():
    for file_name, seizures in patient_data.items():
        for seizure in seizures:
            start, end = seizure
            patients.append(patient)
            files.append(file_name)
            seizure_starts.append(start)
            seizure_ends.append(end)

# DataFrame oluşturalım
df_seizures = pd.DataFrame({
    'patient': patients,
    'file': files,
    'seizure_start': seizure_starts,
    'seizure_end': seizure_ends
})

# Ek olarak seizure_duration sütunu ekleyelim (opsiyonel)
df_seizures['seizure_duration'] = df_seizures['seizure_end'] - df_seizures['seizure_start']

# DataFrame'i görüntüleyelim
print(df_seizures.head())
print(f"\nToplam nöbet sayısı: {len(df_seizures)}")
print(f"Toplam hasta sayısı: {df_seizures['patient'].nunique()}")

df_seizures.to_csv('all_annotations.csv', index=False, encoding='utf-8-sig')

In [ ]:
# drop exclude_list from all_annotations due to channels are too complicated.

exclude_list = [
    'chb12/chb12_29.edf',
    'chb12/chb12_27.edf',
    'chb12/chb12_28.edf'
]

# Sözlüğü güvenli bir şekilde güncellemek için iç içe döngü kullanıyoruz
for item in exclude_list:
    # item örneği: 'chb12/chb12_29.edf'
    # folder: 'chb12', filename: 'chb12_29.edf'
    folder, filename = item.split('/')

    # Eğer bu klasör ana sözlükte varsa ve dosya o klasörün içindeyse sil
    if folder in all_annotations and filename in all_annotations[folder]:
        del all_annotations[folder][filename]

# Sonucu doğrulamak için chb12 klasörüne bakalım
print(all_annotations['chb12'].keys())

In [ ]:
PATIENTS_TO_USE = ['chb01', 'chb03', 'chb06', 'chb10', 'chb14', 'chb15', 'chb16', 'chb24']

for patient_id in PATIENTS_TO_USE:
    file_names = all_annotations[patient_id].keys() # bu şekilde hastanın icindeki edf dosyalarını çekebiliyoruz mesela file_names ilk elemanı chb_01_03.edf
    print(file_names)

In [ ]:
CHB_MIT_PATH = 'data-understanding/data/chb-mit'
FINAL_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ'
]

# for patient_id in PATIENTS_TO_USE:
#     file_names = all_annotations[patient_id].keys() # bu şekilde hastanın icindeki edf dosyalarını çekebiliyoruz mesela file_names ilk elemanı chb_01_03.edf



signals, channel_names, fs = deniz.load_edf_file('data-understanding/data/chb-mit/chb01/chb01_03.edf')
signals2, channel_names2, fs2 = deniz.fix_eeg_channels_load_edf('data-understanding/data/chb-mit/chb01/chb01_03.edf',FINAL_CHANNELS,deniz.load_edf_file)

In [ ]:
signals.shape

In [ ]:
signals2.shape

sinyal, kanal adları ve fs yi kullanarak raw dosyası oluşturuyoruz. bunu da yapmak için info değişkeni oluşturmalıyız, bunu da mne.create_info() fonksiyonu ile yapıyoruz

In [ ]:
info = mne.create_info(ch_names=channel_names, sfreq=fs, ch_types='eeg')

In [ ]:
raw_created = mne.io.RawArray(signals, info)
print(raw_created.info)

In [ ]:
raw_created

In [ ]:
raw = mne.io.read_raw_edf('data-understanding/data/chb-mit/chb01/chb01_03.edf')

In [ ]:
raw.info